In [1]:
import requests

url = ('https://newsapi.org/v2/everything?'
       'q=Apple&'
       'from=2025-04-20&'
       'sortBy=popularity&'
       'apiKey=aa97db5ece6a4d2093fceadcc9f0bf92')

response = requests.get(url)

In [ ]:
from datetime import timedelta
import datetime
from os import environ
try:
    from newsapi.newsapi_client import NewsApiClient
except:
    !pip3 install newsapi.newsapi_client

newsapi = NewsApiClient(api_key='ABC')
to_date = datetime.date.today() - timedelta(days = 1)
from_date = to_date - datetime.timedelta(days=28)

def filter_articles(articles: dict):
    filter_n_articles = articles['articles'][0:len(articles['articles'])]
    list_of_articles = []
    for article in filter_n_articles:
        if "\'" not in article['description'] and 'www.macrumors.com' not in article['url']: 
            a = {
                'source_name': article['source']['name'],
                'author': article['author'],
                'title': article['title'],
                'description': article['description'],
                'published': article['publishedAt'],
                'url': article['url']
            }
        else: continue
        list_of_articles.append(a)
    return list_of_articles


all_articles = newsapi.get_everything(q='apple',
                                       from_param = from_date,
                                       to = to_date,
                                       language = 'en',
                                       sort_by = 'relevancy')
filter_articles(all_articles)

[{'source_name': 'The Verge',
  'author': 'Umar Shakir',
  'title': 'Apple has a new ‘Viral’ playlist on Apple Music and Shazam',
  'description': 'Apple is launching a new global Viral Chart playlist in Apple Music that consists of tracks people are discovering through the company’s Shazam service. The playlist, which is updated daily, shows the top 50 songs people have heard playing out in the real wor…',
  'published': '2025-05-08T21:01:29Z',
  'url': 'https://www.theverge.com/news/663866/apple-music-viral-charts-global-playlist-shazam-top-50'},
 {'source_name': 'The Verge',
  'author': 'Tom Warren, Jess Weatherbed',
  'title': 'The EU isn’t happy with Apple’s tax on alternative app stores',
  'description': 'The European Commission has just issued its first Digital Markets Act (DMA) fines to Apple and Meta, and now it’s telling Apple that it’s not impressed with the company’s approach to alternative app stores. The DMA originally forced Apple to begrudgingly allo…',
  'published': 

In [3]:
all_articles = newsapi.get_everything(q='apple',
                                       from_param = from_date,
                                       to = to_date,
                                       language = 'en',
                                       sort_by = 'relevancy')
print(all_articles)
articles_list = []
filter_n_articles = all_articles['articles'][0:len(all_articles['articles'])]
for article in filter_n_articles:
    if "\'" not in article['description'] and 'www.macrumors.com' not in article['url']:
        articles_list.append(article)

print(len(articles_list))

{'status': 'ok', 'totalResults': 22103, 'articles': [{'source': {'id': 'the-verge', 'name': 'The Verge'}, 'author': 'Umar Shakir', 'title': 'Apple has a new ‘Viral’ playlist on Apple Music and Shazam', 'description': 'Apple is launching a new global Viral Chart playlist in Apple Music that consists of tracks people are discovering through the company’s Shazam service. The playlist, which is updated daily, shows the top 50 songs people have heard playing out in the real wor…', 'url': 'https://www.theverge.com/news/663866/apple-music-viral-charts-global-playlist-shazam-top-50', 'urlToImage': 'https://platform.theverge.com/wp-content/uploads/sites/2/chorus/uploads/chorus_asset/file/23925979/acastro_STK046_03.jpg?quality=90&strip=all&crop=0%2C10.732984293194%2C100%2C78.534031413613&w=1200', 'publishedAt': '2025-05-08T21:01:29Z', 'content': 'Now Apple and Shazam users have a quick place to see what music people are discovering in the real world.\r\nNow Apple and Shazam users have a quick pl

In [4]:
import pandas as pd

filtered_articles = filter_articles(all_articles)
df_articles = pd.DataFrame(filtered_articles)
# df_articles.to_excel('apple_articles.xlsx', index=False)

# print("Saved!")

In [5]:
import asyncio
from crawl4ai import *

async def extract_Article_body(url, baseSelector, selector):
    browser_config = BrowserConfig(
        browser_type="chromium",  # Options: "chromium", "firefox", "webkit"
        headless=True,            # Set to False if you want to see the browser window
        viewport_width=1280,
        viewport_height=800,
        verbose=True,
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    )
    async with AsyncWebCrawler(config = browser_config) as crawler:

        json_strategy = JsonCssExtractionStrategy(
        schema = {
            "name": "Article Body",
            "baseSelector": baseSelector,
            "fields" : [
                {
                "name": "Section",
                "selector": selector,
                "type": "text"
                }
            ]
                },
        verbose = False
        )

        crawler_config = CrawlerRunConfig(
            extraction_strategy = json_strategy,
            screenshot=False,
            verbose=True,
            cache_mode=CacheMode.DISABLED,
            log_console=True,
            # wait_for=".swiper-slide-body",
            scan_full_page=True,  # Tells the crawler to try scrolling the entire page
            scroll_delay=2,     # Delay (seconds) between scroll steps
        )

        result = await crawler.arun(
            url = url, 
            config = crawler_config
        )
        # print(result.markdown)
        return result.extracted_content

In [6]:
async def process_each_article(gizmodo_articles):
    extracted_content_list = []
    for each_url in gizmodo_articles["url"]:
        extracted_content = await extract_Article_body(each_url, "#main > article > div > div", "#main > article > div > div > div > div.mt-5.mx-auto > div.entry-content.prose.dark\:prose-invert.lg\:prose-lg.lg\:leading-\[29px\].font-serif.max-w-none")
        # if not extracted_content:
            # extracted_content = await extract_Article_body(each_url, "#content > article > div.duet--layout--entry-body-container._1t5ltw90._1ymtmqp3._1ymtmqp13 > div", "#zephr-anchor")
        extracted_content_list.append(extracted_content)
    return extracted_content_list

if __name__ == "__main__":
    df_articles = pd.DataFrame(filtered_articles)
    gizmodo_articles = df_articles[df_articles["source_name"] == "Gizmodo.com"]
    extracted_content_list = await process_each_article(gizmodo_articles)
    print(extracted_content_list)


<>:4: SyntaxWarning: invalid escape sequence '\:'
/var/folders/70/3h59ks892kj0r487cwct1kzh0000gn/T/ipykernel_33264/1324094549.py:4: SyntaxWarning: invalid escape sequence '\:'
  extracted_content = await extract_Article_body(each_url, "#main > article > div > div", "#main > article > div > div > div > div.mt-5.mx-auto > div.entry-content.prose.dark\:prose-invert.lg\:prose-lg.lg\:leading-\[29px\].font-serif.max-w-none")


[INIT].... → Crawl4AI 0.6.3 

[CONSOLE]. ℹ Console: JQMIGRATE: Migrate is installed, version 3.4.1

[CONSOLE]. ℹ Console: [[OpenWeb Launcher]] v5.2.1

[CONSOLE]. ℹ Console: Failed to load resource: net::ERR_SOCKET_NOT_CONNECTED

...

[FETCH]... ↓ https://gizmodo.com/apples-second-biggest-suppli...ct-empty-shelves-due-to-trump-tariffs-2000595817  |
✓ | ⏱: 35.66s 

[SCRAPE].. ◆ https://gizmodo.com/apples-second-biggest-suppli...ct-empty-shelves-due-to-trump-tariffs-2000595817  |
✓ | ⏱: 0.15s 

[EXTRACT]. ■ Completed for https://gizmodo.com/apples-second-biggest-supplier... | Time: 0.2788214579995838s 

[COMPLETE] ● https://gizmodo.com/apples-second-biggest-suppli...ct-empty-shelves-due-to-trump-tariffs-2000595817  |
✓ | ⏱: 36.10s 

['[\n    {\n        "Section": "Apple is in the early stages of developing brain-computer interfaces that would allow people, especially those with mobility issues, to control their iPhones, iPads, and Vision Pro headsets with neural signals captured by a new kind of brain implant.1/1SkipAdContinue watchingafter the adVisit Advertiser websiteGO TO PAGEAccording to theWall Street Journal, Apple is partnering with Synchron—a privately held, New York City-based company backed by Jeff Bezos and Bill Gates—on the in project.The brain-computer interface, or BCI, industry is projected to grow significantly over the coming decades.Perhaps the best-known player in the space is Elon Musk’s Neuralink, which, as of January, has successfully implanted its devices in three people.Historically, most users have relied on both mechanical (through the use of a keyboard or mouse) and behavioral (through the use of touchscreens or voice) interaction with their computers. BCIs eliminate the need for physic

In [7]:
import json
from json import loads

gizmodo_articles = gizmodo_articles.reset_index(drop=True)
gizmodo_articles_copied = gizmodo_articles.copy()
extracted_content_listV2 = extracted_content_list.copy()
gizmodo_articles_copied["article_body"] = "Empty"
for i in range(len(extracted_content_listV2)):
    json_article_content = loads(extracted_content_listV2[i])
    # wired_articles_copied["article_body"].loc[i] = json_article_content[0]["Section"]
    # print(i)
    if len(json_article_content) != 0:
        # print(i)
        gizmodo_articles_copied.at[i, "article_body"] = json_article_content[0]["Section"]
    else:
        print("empty " + str(i))
        gizmodo_articles_copied.at[i, "article_body"] = "Empty"

    # wired_articles_copied["article_body"].loc[i] = extracted_content_listV2[i][extracted_content_listV2[i].find(":"):-7]

In [13]:
def split_text_into_chunks(text, max_words):
    words = text.split()
    return [' '.join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

max_words = 5000

new_rows = []
for _, row in gizmodo_articles_copied.iterrows():
    chunks = split_text_into_chunks(row['article_body'], max_words)
    new_row = row.to_dict()
    for i, chunk in enumerate(chunks):
        if i == 0:
            new_row['article_body'] = chunk
        else:
           new_row[f'article_body_part_{i}'] = chunk
    new_rows.append(new_row)

df_expanded = pd.DataFrame(new_rows)

# #Save to Excel
df_expanded.to_excel('apple_Gizmodo_articles_with_body_2.xlsx', index=False)

print("Saved!")

Saved!
